# Calibration quintets (returns cache + SA)

**Goal:** Load daily first/last mid returns from SQLite **once** (or from `data/quintet_returns_cache.pkl`), then rerun **shuffle + simulated annealing** and export JSON **without** querying the DB again.

**Reading a run:** `energy_best` is \(\sum_b \bar r_b^2\) on the **best** layout; `max|bundle mean|` is the worst bundle's |mean log-return|. During SA, printed `E` often rises again while `best` stays flat. This is normal; exports use the **best** state.

**Workflow:** run **Section 1** when the DB or calendar window changes; rerun **Section 2** (and optionally **Section 3**) for new seeds, steps, or cooling.

In [ ]:
from __future__ import annotations

import json
import pickle
import time

import numpy as np

from research_core.scripts.build_calibration_quintets import (
    EVAL_SKIP_FIRST_DAYS,
    EVAL_N_DAYS,
    resolve_data_path,
    load_evaluation_day_keys,
    daily_returns_from_sqlite,
    shuffle_repair_initial_quintets,
    energy_quintets,
    grid_mean_return,
    simulated_annealing,
    simulated_annealing_multiplicity_swaps,
    slot_multiplicities,
    _multiplicity_vector,
    _bundle_means,
)

## 1. Load daily returns (DB once, then pickle cache)

Set `FORCE_REFRESH_FROM_DB = True` to ignore the cache and re-run the large SQLite query.

In [ ]:
DB_REL = "KGHM_order_flow.sqlite"
SKIP = EVAL_SKIP_FIRST_DAYS
N_DAYS = EVAL_N_DAYS
RETURN_MODE = "log"
STRICT_SKIP = False
FORCE_REFRESH_FROM_DB = False  # True -> always query DB

CACHE = resolve_data_path("quintet_returns_cache.pkl")
db_path = resolve_data_path(DB_REL)
assert db_path.is_file(), db_path

cache_key = {
    "db": str(db_path.resolve()),
    "skip": SKIP,
    "n_days": N_DAYS,
    "return_mode": RETURN_MODE,
    "strict_skip": STRICT_SKIP,
}

loaded = False
if not FORCE_REFRESH_FROM_DB and CACHE.is_file():
    with open(CACHE, "rb") as f:
        blob = pickle.load(f)
    if blob.get("cache_key") == cache_key:
        r = np.asarray(blob["r"], dtype=np.float64)
        day_keys = blob["day_keys"]
        skip_effective = int(blob["skip_effective"])
        n_db_days = int(blob["n_db_days"])
        loaded = True
        print("Loaded from cache:", CACHE)

if not loaded:
    t0 = time.time()
    day_keys, skip_effective, n_db_days = load_evaluation_day_keys(
        db_path, SKIP, N_DAYS, strict_skip=STRICT_SKIP
    )
    r = daily_returns_from_sqlite(db_path, day_keys, mode=RETURN_MODE)
    print(f"DB query wall: {time.time() - t0:.2f}s")
    with open(CACHE, "wb") as f:
        pickle.dump(
            {
                "cache_key": cache_key,
                "r": r,
                "day_keys": day_keys,
                "skip_effective": skip_effective,
                "n_db_days": n_db_days,
            },
            f,
            protocol=pickle.HIGHEST_PROTOCOL,
        )
    print("Wrote cache:", CACHE)

n_days = len(day_keys)
n_triple = 500 - 2 * n_days
print("n_days", n_days, "n_triple", n_triple, "n_db_days", n_db_days, "skip_effective", skip_effective)

## 2. Simulated annealing (rerun freely; no DB)

Change `SEED`, `STEPS`, `SLOTS_PER_MOVE` (K cells per greedy proposal), cooling, then run this cell again.

After a run, **Section 2b** can continue from the current `best` layout with new steps and parameters.

In [ ]:
SEED = 0
STEPS = 60_000
T0 = 1
T_MIN = 1e-9
ALPHA = 0.9995
LOG_EVERY = max(1, STEPS // 100)
SLOTS_PER_MOVE = 6  # K distinct grid cells; greedy reassignment of their day ids

rng = np.random.default_rng(SEED)
f = _multiplicity_vector(n_days, n_triple, rng)
triple_mask = f == 3

state0 = shuffle_repair_initial_quintets(f, rng)
e0 = energy_quintets(state0, r)
best, best_e, hist = simulated_annealing(
    state0,
    r,
    rng,
    n_steps=STEPS,
    T0=T0,
    T_min=T_MIN,
    alpha=ALPHA,
    log_every=LOG_EVERY,
    quiet=False,
    slots_per_move=SLOTS_PER_MOVE,
)

means = _bundle_means(best, r)
print(
    "initial E", e0,
    "best E", best_e,
    "max|bundle mean|", float(np.max(np.abs(means))),
)

In [ ]:
import matplotlib.pyplot as plt

out_path = resolve_data_path("calibration_quintets.json")
quintets_keys = [
    [day_keys[int(best[b, j])] for j in range(5)] for b in range(100)
]
triple_day_keys = [day_keys[i] for i in np.where(triple_mask)[0].tolist()]

payload = {
    "db": str(db_path),
    "n_distinct_days_in_db": n_db_days,
    "skip_requested": SKIP,
    "skip_effective": skip_effective,
    "n_days": n_days,
    "day_keys": list(day_keys),
    "return_mode": RETURN_MODE,
    "r_d": r.tolist(),
    "n_triple_slots": int(n_triple),
    "triple_day_keys": triple_day_keys,
    "quintets": quintets_keys,
    "energy_initial": float(e0),
    "energy_final": float(energy_quintets(best, r)),
    "energy_best": float(best_e),
    "bundle_mean_returns": means.tolist(),
    "bundle_mean_abs_max": float(np.max(np.abs(means))),
    "sa": {
        "seed": SEED,
        "steps": STEPS,
        "T0": T0,
        "T_min": T_MIN,
        "alpha": ALPHA,
        "slots_per_move": SLOTS_PER_MOVE,
        "log_every_requested": None,
        "log_every": LOG_EVERY,
        "history_tail": hist[-5:] if len(hist) > 5 else hist,
    },
}

with open(out_path, "w", encoding="utf-8") as fp:
    json.dump(payload, fp, indent=2)
print("Wrote", out_path)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(np.arange(100), means, color="steelblue", alpha=0.85)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Quintet index")
ax.set_ylabel("Mean daily return (log)")
ax.set_title("Per-quintet mean return (best SA state)")
plt.tight_layout()
plt.show()

## 2b. Multiplicity moves + optional slot SA

Run **Section 2** first so `best`, `best_e`, `hist`, `rng`, and `n_days` exist.

**Phase A (multiplicity):** Metropolis moves that change **which** calendar days are **3× vs 2×** by replacing one cell occupied by a triple-count day with a double-count day (row must stay 5 distinct days). Rows with larger **|**bundle mean**|** are sampled more often (`MULT_ROW_BIAS_POWER`); `MULT_UNIFORM_MIX` keeps uniform exploration.

`MULT_METROPOLIS_OBJECTIVE`: `"energy"` (default) = minimize \(\sum_b \bar r_b^2\); `"grid_mean"` = minimize the **mean** of `r` over all 500 cells (pulls average return down when it is too high); `"abs_grid_mean"` = minimize **|**that mean**|** (toward zero).

**Phase B (optional):** If `CONTINUE_STEPS > 0`, run slot `simulated_annealing` from the result with your usual cooling and `CONTINUE_SLOTS` (K).

In [ ]:
# Requires Section 2: best, best_e, hist, rng, r, n_days
CONTINUE_SEED = None  # int for a fresh RNG; None = keep using `rng` from Section 2

# --- Phase A: change which days are 3× vs 2× ---
MULT_STEPS = 50_000
MULT_T0 = 1e-12
MULT_T_MIN = 1e-12
MULT_ALPHA = 0.9995
MULT_LOG_EVERY = max(1, MULT_STEPS // 100)
MULT_ROW_BIAS_POWER = 2.0  # higher -> focus more on high-|mean| quintets
MULT_UNIFORM_MIX = 0.15  # fraction of proposals picking a uniformly random row
# "energy" | "grid_mean" (lower average return) | "abs_grid_mean" (|mean| -> 0)
MULT_METROPOLIS_OBJECTIVE = "energy"

# --- Phase B: optional slot SA (set steps to 0 to skip) ---
CONTINUE_STEPS = 0
CONTINUE_T0 = 1e-3
CONTINUE_T_MIN = 1e-10
CONTINUE_ALPHA = 0.9995
CONTINUE_LOG_EVERY = max(1, CONTINUE_STEPS // 100) if CONTINUE_STEPS else 1
CONTINUE_SLOTS = 8  # K cells per slot proposal (2 .. 500)

rng_c = rng if CONTINUE_SEED is None else np.random.default_rng(int(CONTINUE_SEED))
_start = best.copy()
_e_start = float(energy_quintets(_start, r))

best, best_e, hist_mult = simulated_annealing_multiplicity_swaps(
    _start,
    r,
    rng_c,
    n_days,
    n_steps=MULT_STEPS,
    T0=MULT_T0,
    T_min=MULT_T_MIN,
    alpha=MULT_ALPHA,
    log_every=MULT_LOG_EVERY,
    quiet=False,
    row_bias_power=MULT_ROW_BIAS_POWER,
    uniform_mix=MULT_UNIFORM_MIX,
    metropolis_objective=MULT_METROPOLIS_OBJECTIVE,
)
hist.extend(hist_mult)

f = slot_multiplicities(best, n_days)
triple_mask = f == 3

if CONTINUE_STEPS > 0:
    best, best_e, hist_cont = simulated_annealing(
        best,
        r,
        rng_c,
        n_steps=CONTINUE_STEPS,
        T0=CONTINUE_T0,
        T_min=CONTINUE_T_MIN,
        alpha=CONTINUE_ALPHA,
        log_every=CONTINUE_LOG_EVERY,
        quiet=False,
        slots_per_move=int(CONTINUE_SLOTS),
    )
    hist.extend(hist_cont)
    SLOTS_PER_MOVE = int(CONTINUE_SLOTS)
# else: keep SLOTS_PER_MOVE from Section 2 for JSON

means = _bundle_means(best, r)
print(
    "2b: start_E",
    _e_start,
    "best_E_or_score",
    best_e,
    "grid_mean@best",
    grid_mean_return(best, r),
    "max|bundle mean|",
    float(np.max(np.abs(means))),
)

## 3. Export JSON + quick plot

Re-run after **Section 2** or **2b** so `best` and `means` match what you export. (Section 2b sets `SLOTS_PER_MOVE` to the last continuation’s K for the JSON block.)

In [ ]:
out_path = resolve_data_path("calibration_quintets.json")
quintets_keys = [
    [day_keys[int(best[b, j])] for j in range(5)] for b in range(100)
]
triple_day_keys = [day_keys[i] for i in np.where(triple_mask)[0].tolist()]

payload = {
    "db": str(db_path),
    "n_distinct_days_in_db": n_db_days,
    "skip_requested": SKIP,
    "skip_effective": skip_effective,
    "n_days": n_days,
    "day_keys": list(day_keys),
    "return_mode": RETURN_MODE,
    "r_d": r.tolist(),
    "n_triple_slots": int(n_triple),
    "triple_day_keys": triple_day_keys,
    "quintets": quintets_keys,
    "energy_initial": float(e0),
    "energy_final": float(energy_quintets(best, r)),
    "energy_best": float(best_e),
    "bundle_mean_returns": means.tolist(),
    "bundle_mean_abs_max": float(np.max(np.abs(means))),
    "sa": {
        "seed": SEED,
        "steps": STEPS,
        "T0": T0,
        "T_min": T_MIN,
        "alpha": ALPHA,
        "slots_per_move": SLOTS_PER_MOVE,
        "log_every_requested": None,
        "log_every": LOG_EVERY,
        "history_tail": hist[-5:] if len(hist) > 5 else hist,
    },
}

with open(out_path, "w", encoding="utf-8") as fp:
    json.dump(payload, fp, indent=2)
print("Wrote", out_path)

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(np.arange(100), means, color="steelblue", alpha=0.85)
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("Quintet index")
ax.set_ylabel("Mean daily return (log)")
ax.set_title("Per-quintet mean return (best SA state)")
plt.tight_layout()
plt.show()

## 4. Layout sanity checks

Confirm the exported layout is valid: every evaluation day appears 2 or 3 times
on the 100x5 grid, each quintet has 5 distinct days, and no quintet's mean daily
log return exceeds the cap.

In [ ]:
# Requires Section 2 (and optional 2b): `best`, `r`, `n_days`
# Checks: (1) each evaluation day appears exactly 2 or 3 times on the 100×5 grid;
#         (2) each quintet row has 5 distinct day indices;
#         (3) every quintet's mean daily log return has |mean| < 4e-5.

import numpy as np

MEAN_ABS_CAP = 4e-5

f = slot_multiplicities(best, n_days)
bad_mult = [(int(i), int(f[i])) for i in range(n_days) if int(f[i]) not in (2, 3)]

bad_rows = [int(b) for b in range(100) if np.unique(best[b]).size != 5]

means = _bundle_means(best, r)
bad_means = [(int(b), float(means[b])) for b in range(100) if abs(float(means[b])) >= MEAN_ABS_CAP]

ok_mult = len(bad_mult) == 0
ok_rows = len(bad_rows) == 0
ok_means = len(bad_means) == 0

print("=== Calibration layout checks ===")
print(f"  multiplicities ∈ {{2,3}} for all {n_days} days: {ok_mult}")
if bad_mult:
    print(f"    violations (day_index, count), first 20: {bad_mult[:20]}")
print(f"  5 distinct days per quintet (100 rows): {ok_rows}")
if bad_rows:
    print(f"    bad row indices (first 20): {bad_rows[:20]}")
print(f"  |quintet mean log-return| < {MEAN_ABS_CAP:g} for all 100: {ok_means}")
if bad_means:
    print(f"    violations (quintet_index, mean), first 20: {bad_means[:20]}")
print(f"  max |bundle mean| = {float(np.max(np.abs(means))):.6g}")

assert ok_mult and ok_rows and ok_means, "One or more checks failed (see prints above)."
print("All checks passed.")